# 00 Smoke test: is the environment working?

Run this at the start of Day 1 (step 4 of the setup sheet). Click **Run All**, then save (Ctrl+S). Every cell should finish without a red error.
It checks Python and the main libraries, trains a tiny model on scikit-learn's California Housing data,
and logs one run to MLflow, which we open on Day 2.

In [ ]:
import sys
import numpy, pandas, sklearn, matplotlib, mlflow

print("Python      ", sys.version.split()[0])
for lib in (numpy, pandas, sklearn, matplotlib, mlflow):
    print(f"{lib.__name__:<12}", lib.__version__)

## 1. Load a built-in dataset

`fetch_california_housing` downloads a small file the first time. If the network blocks it,
the cell falls back to a synthetic dataset so the rest of the test still runs.

In [ ]:
from sklearn.datasets import fetch_california_housing, make_regression
import pandas as pd

try:
    X, y = fetch_california_housing(return_X_y=True, as_frame=True)
    source = "California Housing"
except Exception as err:  # no internet, blocked download, etc.
    print("Download failed, using synthetic data instead:", err)
    X_arr, y = make_regression(n_samples=2000, n_features=8, noise=10, random_state=42)
    X = pd.DataFrame(X_arr, columns=[f"f{i}" for i in range(8)])
    source = "synthetic"

print(source, X.shape)
X.head()

## 2. Train and score a tiny model

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = LinearRegression().fit(X_train, y_train)
r2 = r2_score(y_test, model.predict(X_test))
print(f"R² on the test set: {r2:.3f}")

## 3. Make a chart (checks matplotlib works in the browser)

In [ ]:
import matplotlib.pyplot as plt

preds = model.predict(X_test)
plt.figure(figsize=(5, 4))
plt.scatter(y_test, preds, s=4, alpha=0.4)
plt.xlabel("actual")
plt.ylabel("predicted")
plt.title(f"Smoke test ({source})")
plt.show()

## 4. Log the run to MLflow

This writes to a folder called `mlruns/` in the repo root. On Day 2 MLflow becomes our lab notebook.

In [ ]:
from pathlib import Path
import mlflow

ROOT = Path.cwd() if (Path.cwd() / "requirements.txt").exists() else Path.cwd().parent
mlflow.set_tracking_uri((ROOT / "mlruns").as_uri())
mlflow.set_experiment("smoke-test")

with mlflow.start_run(run_name="setup-check"):
    mlflow.log_param("model", "LinearRegression")
    mlflow.log_param("data", source)
    mlflow.log_metric("r2", r2)

print("Logged to", mlflow.get_tracking_uri())

## 5. Open the MLflow UI (Day 2, not today)

On Day 2, in the **terminal** (not here), from the repo root, run:

```bash
mlflow ui --port 5000
```

Codespaces shows a pop-up "Open in Browser" for port 5000 (or open the **Ports** tab).
You should see the experiment `smoke-test` with one run called `setup-check` and its `r2` metric.
Press `Ctrl+C` in the terminal to stop the UI afterwards.

**All cells ran without a red error? The environment is ready for Day 1.** Save the notebook (Ctrl+S).